In [1]:
try:
    import lightgbm as lgb
except:
    !uv pip install -q lightgbm
    import lightgbm as lgb

print(f"lightgbm version: {lgb.__version__}")

lightgbm version: 4.6.0


In [2]:
## -- System dependencies --
import sys, os, gc
import torch

## -- Device-Agnostic (GPU/CPU) --
def get_system_info():
    if torch.cuda.is_available():
        return f"GPU: {torch.cuda.get_device_name(0)}"
    else:
        return f"CPU: {os.cpu_count()}"

get_system_info()

'CPU: 4'

In [3]:
try:
    from colorama import Fore, Style
except:
    !pip install -q colorama
    from colorama import Fore, Style

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

In [4]:
## -- Data Manipulation --
import numpy as np, pandas as pd, random

## -- Visualization --
from IPython.display import display, Image
import matplotlib.pyplot as plt
import seaborn as sns

## -- Functional Tools --
import itertools
from tqdm.auto import tqdm
from time import time, sleep

## -- Machine Learning --
from sklearn.calibration import CalibrationDisplay
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder as skTE
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')

## -- Global Settings --
# sklearn.set_config(transform_output='pandas')

# pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', 100)
# sns.set_style("whitegrid")
# plt.style.use('ggplot')

In [5]:
TEAL, GOLD, CORAL, BLUE, DARK, CRIM, MUTED, INK, PAPER = '#168c91','#e8b44f','#dc705a',\
                                                         'dodgerblue','#0B4F7A','crimson',\
                                                         '#627D98','#102A43','#faf9f5'
plt.rcParams.update({
    'figure.facecolor': PAPER,
    'axes.facecolor': PAPER,
    'savefig.facecolor': PAPER,
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 11,
    'axes.titleweight': 'semibold',
    'axes.labelcolor': INK,
    'text.color': INK,
    'xtick.color': DARK,
    'ytick.color': DARK,
    'axes.edgecolor': MUTED,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.color': MUTED,
    'patch.edgecolor': DARK,
    # 'patch.force_edgecolor': True,
    # 'patch.linewidth': 1.5,
    # 'figure.dpi':110,
    # 'savefig.dpi':160,
})
# CMAP = LinearSegmentedColormap.from_list('atlas', ['#f0f5ef','#a9d6ca',TEAL,INK])
# sns.set_palette([TEAL, GOLD, CORAL, BLUE, MUTED, CRIM, DARK, INK, PAPER])

## -- Set Global Seed --
CFG = {
    'FOLDS':  5,
    'SEED':   42,
    'GREEN':  '\033[32m',
    'YELLOW': '\033[33m',
    'RESET':  '\033[0m'
}

print(f"CLASSIC {CFG['GREEN']} GREEN {CFG['RESET']} {CFG['YELLOW']} YELLOW {CFG['RESET']}")

CLASSIC  GREEN   YELLOW 


In [6]:
## -- ⚠️ IMPORTANT: SELECT PLATFORM ⚠️ --
PLATFORM = "kaggle" # -> 'colab' 'kaggle'

if PLATFORM == "kaggle":
    PATH   = "/kaggle/input/competitions/playground-series-s6e11/"
    submit = pd.read_csv(PATH+"sample_submission.csv")
    train  = pd.read_csv(PATH+"train.csv")
    test   = pd.read_csv(PATH+"test.csv")
    ORIG_PATH = "/kaggle/input/datasets/uditjain13/credit-card-fraud-detection-2026/"
    orig = pd.read_csv(ORIG_PATH+"credit_card_fraud_2026.csv")
elif PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PATH   = "/content/drive/MyDrive/-- shared_notebooks --/Ps6e11 | Credit Card Fraud/_fraud_data/"
    submit = pd.read_csv(PATH+"sample_submission.csv")
    train  = pd.read_csv(PATH+"train.csv")
    test   = pd.read_csv(PATH+"test.csv")
    orig   = pd.read_csv(PATH+"credit_card_fraud_2026.csv")
    
## =================================================================================

for (name, df) in {"Train": train, "Test": test, 'Orig': orig}.items(): #
    print(f"{name} shape: {df.shape}")

print(f"\nFrame type: {type(train)}")

float_cols = [
    'amount_usd', 'account_balance_usd',
    'hours_since_last_txn', 'distance_from_home_km',
    'velocity_score', 'merchant_risk_score',
]

int_cols = [
    'customer_age', 'card_age_months', 'time_of_day_hour', 'txn_count_last_24h',
    'cvv_retry_count', 'day_of_week', 'prior_disputes',
]

train.head()

Train shape: (699960, 26)
Test shape: (299984, 25)
Orig shape: (20000, 26)

Frame type: <class 'pandas.core.frame.DataFrame'>


,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,merchant_risk_score,prior_disputes,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,17,3,29.2,0,Groceries,Mastercard,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,23,5,29.9,1,Online Retail,Visa,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,15,0,24.1,0,Utilities,Visa,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,7,3,41.9,0,Groceries,Mastercard,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,4,4,48.0,0,Electronics,Visa,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [7]:
train_len = len(train)
test_len  = len(test)

n_unique = train.nunique().sort_values()
n_unique

is_new_merchant                      2
is_fraud                             2
is_ai_generated_scam_attempt         2
billing_shipping_mismatch            2
ip_country_mismatch                  2
used_vpn                             2
is_foreign_transaction               2
cvv_retry_count                      4
auth_method                          5
prior_disputes                       5
card_type                            5
channel                              5
day_of_week                          7
device_type                          8
txn_count_last_24h                  12
merchant_category                   12
time_of_day_hour                    24
card_age_months                     49
customer_age                        64
velocity_score                     666
merchant_risk_score                950
hours_since_last_txn              3633
distance_from_home_km             4365
amount_usd                       13387
account_balance_usd              43317
id                       

In [8]:
# import pandas as pd
# from pandas.util import hash_pandas_object

# # Generate hashes
# hash1 = hash_pandas_object(train[base_cols], index=False)
# hash2 = hash_pandas_object(orig[base_cols], index=False)

# # Find rows in orig that are NOT present in train
# new_or_changed_rows = hash2.isin(hash1)

# # Get the actual data for those mismatched rows
# mismatched_orig_data = orig[new_or_changed_rows]
# display(mismatched_orig_data)

In [9]:
## -- Groupby features FE -- ⚠️
from typing import List, Optional, Sequence
from sklearn.base import BaseEstimator, TransformerMixin

class GroupbyAggregator(BaseEstimator, TransformerMixin):
    """
    Leak-free groupby feature generator for CV folds and multiple test sets.
    Fits aggregations on training data only and left-joins them onto any DataFrame.
    """

    def __init__(
        self,
        grouper: str,
        agg_cols: List[str],
        aggs: Sequence[str] = ("mean", "std"),
    ):
        self.grouper = grouper
        self.agg_cols = [c for c in agg_cols if c != grouper]
        self.aggs = aggs
        self.stats_: Optional[pd.DataFrame] = None
        self.group_cols_: List[str] = []

    def fit(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> "GroupbyAggregator":
        """Compute group aggregations strictly on X (e.g., X_train_fold)."""
        target_cols = [col for col in self.agg_cols if col != self.grouper]
        if not target_cols:
            self.stats_ = None
            self.group_cols_ = []
            return self

        prefix = self.grouper.split("_")[0]
        stats = X.groupby(self.grouper, observed=False)[target_cols].agg(list(self.aggs))

        # Flatten MultiIndex columns into clean feature names
        self.group_cols_ = [
            f"{col}_{prefix}_GROUPBY_{agg}".replace(" ", "")
            for col, agg in stats.columns
        ]
        stats.columns = self.group_cols_
        self.stats_ = stats
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Map learned group statistics onto X while preserving its original index."""
        if self.stats_ is None or not self.group_cols_:
            return X.copy()

        # Left merge preserves row order; re-assign original index to prevent resets
        merged = X.merge(self.stats_, on=self.grouper, how="left")
        merged.index = X.index
        return merged

    def transform_many(self, *dfs: pd.DataFrame) -> List[pd.DataFrame]:
        """Convenience helper to transform multiple DataFrames in a single call."""
        return [self.transform(df) for df in dfs]


In [10]:
from sklearn.base import BaseEstimator, TransformerMixin ## ===== Target/Category Mean ENCODERS =====

class customTE(BaseEstimator, TransformerMixin):
    """
    Target Encoder that supports multiple aggregation functions,
    internal cross-validation for leakage prevention, and smoothing.

    Parameters
    ----------
    cols_to_encode : list of str
        List of column names to be target encoded.

    aggs : list of str, default=['mean']
        List of aggregation functions to apply. Any function accepted by
        pandas' `.agg()` method is supported, such as:
        'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique',
        'count', 'sum', 'median'.
        Smoothing is applied only to the 'mean' aggregation.

    cv : int, default=5
        Number of folds for cross-validation in fit_transform.

    smooth : float or 'auto', default='auto'
        The smoothing parameter `m`. A larger value puts more weight on the
        global mean. If 'auto', an empirical Bayes estimate is used.

    drop_original : bool, default=False
        If True, the original columns to be encoded are dropped.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """
        Learn mappings from the entire dataset.
        These mappings are used for the transform method on validation/test data.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # Learn global statistics for each aggregation
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # Learn category-specific mappings
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        """
        Apply learned mappings to the data.
        Unseen categories are filled with global statistics.
        """
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

    def fit_transform(self, X, y):
        """
        Fit and transform the data using internal cross-validation to prevent leakage.
        """
        # First, fit on the entire dataset to get global mappings for transform method
        self.fit(X, y)

        # Initialize an empty DataFrame to store encoded features
        encoded_features = pd.DataFrame(index=X.index)

        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_trn, y_trn = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_trn.copy()
            temp_df_train['target'] = y_trn

            for col in self.cols_to_encode:
                # --- Calculate mappings only on the training part of the fold ---
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'

                    # Calculate global stat for this fold
                    fold_global_stat = y_trn.agg(agg_func)

                    # Calculate category stats for this fold
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    # --- Apply smoothing only for 'mean' aggregation ---
                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()

                        m = self.smooth
                        if self.smooth == 'auto':
                            # Empirical Bayes smoothing
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0  # No smoothing if no variance between groups

                        # Apply smoothing formula
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)

                    # Store encoded values for the validation fold
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # Merge with original DataFrame
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed


class CategoryMeanTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols=None):
        self.cat_cols = cat_cols
        self.mappings_ = {}
    def fit(self, X, y):
        X = X.copy()
        if self.cat_cols is None:
            self.cat_cols = X.select_dtypes(include=['category']).columns.tolist()
        self.mappings_ = {}
        for col in self.cat_cols:
            df_temp = pd.DataFrame({col: X[col], 'y': y})
            group_means = df_temp.groupby(col, dropna=False)['y'].mean()
            sorted_categories = group_means.sort_values().index
            self.mappings_[col] = {cat: i for i, cat in enumerate(sorted_categories)}

        return self

    def transform(self, X, y=None):
        X = X.copy()
        for col, mapping in self.mappings_.items():
            if col in X.columns:
                X[col] = X[col].map(mapping)

        return X

In [11]:
## -- original_TE with factorization --
from typing import List, Optional, Tuple

def original_fcTE(
    orig: pd.DataFrame,
    df: pd.DataFrame,
    features: Optional[List[str]] = None,
    target: Optional[str] = None,
    aggs: Optional[List[str]] = None,
    fill_nan: bool = False,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Applies out-of-fold / external dataset target encoding (TE) from `orig` onto `df`.
    """
    X_df = df.copy()
    orig_df = orig.copy()  # Prevent mutating the caller's original DataFrame

    # 1. Early exit with consistent return signature (DataFrame, list)
    if not features or not aggs or target is None:
        return X_df, []

    if target not in orig_df.columns:
        raise ValueError(f"Target column '{target}' not found in `orig`.")

    # 2. Validate features exist in BOTH DataFrames
    valid_features = [
        col for col in features 
        if col in orig_df.columns and col in X_df.columns
    ]
    
    # Separate 'count' from target-dependent aggregations
    supported_target_aggs = {'mean', 'median', 'nunique', 'std', 'skew', 'max', 'min'}
    clean_aggs = [a.lower() for a in aggs]

    orig_cols = []
    new_features_dict = {}

    for col in tqdm(valid_features, desc="original_augmenting"):
        # Align categorical/object/bool dtypes safely only for needed columns
        if pd.api.types.is_object_dtype(X_df[col]) or \
           isinstance(X_df[col].dtype, pd.CategoricalDtype) or \
           pd.api.types.is_bool_dtype(X_df[col]):
            combined = pd.concat([X_df[col], orig_df[col]], ignore_index=True)
            codes, _ = pd.factorize(combined, sort=True)
            X_df[col] = codes[:len(X_df)]
            orig_df[col] = codes[len(X_df):]

        # Group once per feature instead of once per (feature, agg)
        grouped = orig_df.groupby(col, dropna=False)
        
        # Compute target aggregations in one vectorized pass
        target_aggs_to_run = [a for a in clean_aggs if a in supported_target_aggs]
        if target_aggs_to_run:
            try:
                agg_stats = grouped[target].agg(target_aggs_to_run)
                for agg_key in target_aggs_to_run:
                    new_col = f"oTE_{col}_{agg_key}"
                    # .map() preserves X_df's exact index and row order
                    new_features_dict[new_col] = X_df[col].map(agg_stats[agg_key])
                    orig_cols.append(new_col)
            except Exception as e:
                print(f"Warning: failed target aggs for col={col}: {e}")

        # Handle 'count' separately since it doesn't depend on `target`
        if 'count' in clean_aggs:
            new_col = f"oTE_{col}_count"
            counts = grouped.size()
            new_features_dict[new_col] = X_df[col].map(counts)
            orig_cols.append(new_col)

    # Concatenate all new columns at once to avoid DataFrame fragmentation
    if new_features_dict:
        new_cols_df = pd.DataFrame(new_features_dict, index=X_df.index)
        X_df = pd.concat([X_df, new_cols_df], axis=1)

    # 3. Fill missing values cleanly
    if fill_nan and orig_cols:
        global_mean = orig_df[target].mean()
        global_median = orig_df[target].median()

        fill_values = {}
        for c in orig_cols:
            if c.endswith(('_mean', '_max', '_min')):
                fill_values[c] = global_mean
            elif c.endswith('_median'):
                fill_values[c] = global_median
            else:
                fill_values[c] = 0

        X_df = X_df.fillna(value=fill_values)

    return X_df, orig_cols

print('-- Factorized augment function ready! --')

-- Factorized augment function ready! --


In [12]:
## -- original_TE with objects --
from typing import List, Optional, Tuple

def original_obTE(
    orig: pd.DataFrame,
    df: pd.DataFrame,
    features: Optional[List[str]] = None,
    target: Optional[str] = None,
    aggs: Optional[List[str]] = None,
    fill_nan: bool = False,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Applies external dataset target encoding (TE) from `orig` onto `df`
    while preserving original categorical/string dtypes.
    """
    X_df = df.copy()

    if not features or not aggs or target is None:
        return X_df, []

    if target not in orig.columns:
        raise ValueError(f"Target column '{target}' not found in `orig`.")

    valid_features = [
        col for col in features 
        if col in orig.columns and col in X_df.columns
    ]
    
    supported_target_aggs = {'mean', 'median', 'nunique', 'std', 'skew', 'max', 'min'}
    clean_aggs = [a.lower() for a in aggs]

    orig_cols = []
    new_features_dict = {}

    for col in tqdm(valid_features, desc="original_augmenting"):
        # Group directly on raw strings, objects, or categories (observed=True prevents cartesian blowups)
        grouped = orig.groupby(col, dropna=False, observed=True)
        
        target_aggs_to_run = [a for a in clean_aggs if a in supported_target_aggs]
        if target_aggs_to_run:
            try:
                agg_stats = grouped[target].agg(target_aggs_to_run)
                for agg_key in target_aggs_to_run:
                    new_col = f"oTE_{col}_{agg_key}"
                    # Convert Series to plain dict so CategoricalIndex mismatches don't cause issues
                    mapping_dict = agg_stats[agg_key].to_dict()
                    new_features_dict[new_col] = X_df[col].map(mapping_dict).astype('float32')
                    orig_cols.append(new_col)
            except Exception as e:
                print(f"Warning: failed target aggs for col={col}: {e}")

        if 'count' in clean_aggs:
            new_col = f"oTE_{col}_count"
            counts_dict = grouped.size().to_dict()
            new_features_dict[new_col] = X_df[col].map(counts_dict).astype('float32')
            orig_cols.append(new_col)

    if new_features_dict:
        new_cols_df = pd.DataFrame(new_features_dict, index=X_df.index)
        X_df = pd.concat([X_df, new_cols_df], axis=1)

    if fill_nan and orig_cols:
        global_mean = orig[target].mean()
        global_median = orig[target].median()

        fill_values = {}
        for c in orig_cols:
            if c.endswith(('_mean', '_max', '_min')):
                fill_values[c] = global_mean
            elif c.endswith('_median'):
                fill_values[c] = global_median
            else:
                fill_values[c] = 0

        X_df = X_df.fillna(value=fill_values)

    return X_df, orig_cols

print('-- Object augment function ready! --')

-- Object augment function ready! --


## FEATURE ENGINEERING

In [13]:
# ## -- Cyclic encoding --
# cyclic_cols = []
# cy_cols = ['time_of_day_hour'] # 'txn_count_last_24h'

# for col in tqdm(cy_cols):
#     for p in [24]:
#         n_s = f"{col}_SINE_{p}"
#         train[n_s] = np.sin(2 * np.pi * train[col] / p).astype('float32')
#         test[n_s]  = np.sin(2 * np.pi * test[col] / p).astype('float32')
#         orig[n_s]  = np.sin(2 * np.pi * orig[col] / p).astype('float32')
#         cyclic_cols.append(n_s)

#         # n_c = f"{col}_COSINE_{p}"
#         # train[n_c] = np.cos(2 * np.pi * train[col] / p).astype('float32')
#         # test[n_c] = np.cos(2 * np.pi * test[col] / p).astype('float32')
#         # orig[n_c] = np.cos(2 * np.pi * orig[col] / p).astype('float32')
#         # CYCLICS.append(n_c)

# # NUMS.extend(CYCLICS)

# print(f"Cylcic Features created: {len(cyclic_cols)}")
# display(train[cyclic_cols].describe().T)

# train[cyclic_cols].head()

In [14]:
ID = 'id'
TARGET = 'is_fraud'

X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET].astype(int)
X_test = test.drop([ID], axis=1); test_id = test[ID]

# del train, test
print('-'*40)
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()
base_cols = cat_cols + num_cols
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('merchant_category', 'card_type', 'txn_count_last_24h'),
    ('merchant_category', 'customer_age'),
]
def feature_engineering(df, fit=False):
    # Digit extraction
    for col in ['amount_usd', 'hours_since_last_txn',
                'velocity_score', 'distance_from_home_km']:
        decimal_name = f"_{col}_decimal"
        df[decimal_name] = (df[col] % 1).round(2).astype('float32')

    # Categorize numericals
    for col in num_cols + ['_amount_usd_decimal', '_hours_since_last_txn_decimal',
                           '_velocity_score_decimal', '_distance_from_home_km_decimal']:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        values = np.floor(df[col]) if col in num_cols else df[col]
        if fit:
            codes, uniques = values.factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = values.map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        # df[cat_name] = df[cat_name].astype('category')

    # Discretize numericals
    bin_config = {'merchant_risk_score': [7], 'hours_since_last_txn': [500, 1000]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                # df[bin_name] = df[bin_name].astype('category')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        # df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape)
print('-'*40)

----------------------------------------
X      init shape: (699960, 24)
X_test init shape: (299984, 24) 

init len(cat_cols): 11
init len(num_cols): 13 

len(new_cat_cols): 22
len(new_num_cols): 4 

prep len(cat_cols): 33
prep len(num_cols): 17 

X      prep shape: (699960, 50)
X_test prep shape: (299984, 50)
----------------------------------------


In [15]:
# USE_FULL_TRAIN = True

# x_sample, x_sample2 = train_test_split(
#     train,
#     train_size=300_000,
#     stratify=train[TARGET],
#     random_state=42
# )

# train_X = train.copy() if USE_FULL_TRAIN else x_sample.reset_index(drop=True)
# print(train_X.shape, type(train_X))

# FEATURES = [c for c in train_X.columns if c not in [ID, TARGET]]

In [16]:
# ## -- Handle optional original table aggregations
# USE_ORIGINAL = False

# import_cols = base_cols

# if USE_ORIGINAL: # try 'std' again later ⚠️
#     aggs_ = ['mean', 'count']  # "mean", "count", "std", "skew", "nunique",
#     feats = FEATURES.copy()
#     x_train, orig_cols = original_obTE(
#         orig, train_X, target=TARGET, features=import_cols,
#         aggs=aggs_, #fill_nan=True
#     )
#     x_test, _ = original_obTE(
#         orig, test, target=TARGET, features=import_cols,
#         aggs=aggs_, #fill_nan=True
#     )
#     feats.extend(orig_cols)
#     X = x_train[feats]
#     X_test = x_test[feats]
#     y = train_X[TARGET].astype("int8")
# else:
#     X = train_X[FEATURES]
#     X_test = test[FEATURES]
#     y = train_X[TARGET].astype("int8")

# print(X.shape, type(X))
# X.head()

In [17]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 50 columns):
 #   Column                                           Non-Null Count   Dtype  
---  ------                                           --------------   -----  
 0   amount_usd                                       699960 non-null  float64
 1   hours_since_last_txn                             699960 non-null  float64
 2   txn_count_last_24h                               699960 non-null  int64  
 3   distance_from_home_km                            699960 non-null  float64
 4   card_age_months                                  699960 non-null  int64  
 5   customer_age                                     699960 non-null  int64  
 6   account_balance_usd                              699960 non-null  float64
 7   cvv_retry_count                                  699960 non-null  int64  
 8   velocity_score                                   699960 non-null  float64
 9   time_of_day_hou

In [18]:
## -- Reduce data memory --

def reduce_memory(df, factorize_first=False):
    # Use copy() if you want to avoid modifying the original dataframe in place
    # df = df.copy()

    old_size = df.memory_usage(deep=True).sum() / (1024 * 1024)

    for c in tqdm(df.columns, desc='Reducing memory...'):
        col_type = df[c].dtype
        
        # If it is already a category, we don't need to reduce it further
        if isinstance(col_type, pd.CategoricalDtype):
            continue
            
        # Check if type is bool (True or False)
        if col_type == bool:
            if factorize_first:
                df[c], _ = df[c].factorize(sort=True)
                df[c] = df[c].astype('category')
            else:
                df[c] = df[c].astype(str).astype('category')
        # Check if type is integer
        elif np.issubdtype(col_type, np.integer):
            df[c] = pd.to_numeric(df[c], downcast='integer')
        # Optional: Convert object types with low cardinality to category
        elif col_type == object:
            if df[c].nunique() < len(df[c]) * 0.5:  # If less than 50% unique values
                if factorize_first:
                    df[c], _ = df[c].factorize(sort=True)
                    df[c] = df[c].astype('category')
                else:
                    df[c] = df[c].astype('category')
        # Check if type is float
        # elif np.issubdtype(col_type, np.floating):
        #     df[c] = pd.to_numeric(df[c], downcast='float')

    new_size = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"Size before process: {old_size:.1f} MB\nSize after process : {new_size:.1f} MB\n")

    return df

X = reduce_memory(X, factorize_first=False)
X_test = reduce_memory(X_test, factorize_first=False)
# orig = reduce_memory(orig)

print(f"Data memory reduced!")
gc.collect()

X.head()

Reducing memory...:   0%|          | 0/50 [00:00<?, ?it/s]

Size before process: 383.0 MB
Size after process : 73.4 MB



Reducing memory...:   0%|          | 0/50 [00:00<?, ?it/s]

Size before process: 142.4 MB
Size after process : 31.5 MB

Data memory reduced!


,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,merchant_risk_score,prior_disputes,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,_amount_usd_decimal,_hours_since_last_txn_decimal,_velocity_score_decimal,_distance_from_home_km_decimal,amount_usd_cat_,hours_since_last_txn_cat_,txn_count_last_24h_cat_,distance_from_home_km_cat_,card_age_months_cat_,customer_age_cat_,account_balance_usd_cat_,cvv_retry_count_cat_,velocity_score_cat_,time_of_day_hour_cat_,day_of_week_cat_,merchant_risk_score_cat_,prior_disputes_cat_,amount_usd_decimal_cat_,hours_since_last_txn_decimal_cat_,velocity_score_decimal_cat_,distance_from_home_km_decimal_cat_,merchant_risk_score_7_quantile_bin_,hours_since_last_txn_500_quantile_bin_,hours_since_last_txn_1000_quantile_bin_,merchant_category_card_type_txn_count_last_24h_,merchant_category_customer_age_
0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,17,3,29.2,0,Groceries,Mastercard,OTP,POS,iPhone,False,False,False,False,False,False,0.16,0.77,0.5,0.78,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,371,553,0,0
1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,23,5,29.9,1,Online Retail,Visa,PIN,In-App,Android Phone,False,False,False,False,False,False,0.83,0.54,0.4,0.06,1,1,1,1,1,1,1,0,1,1,1,0,1,1,1,1,1,2,80,104,1,1
2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,15,0,24.1,0,Utilities,Visa,OTP,Online,Tablet,False,False,True,False,False,False,0.36,0.20,0.8,0.17,2,1,2,2,2,2,2,0,2,2,2,1,0,2,2,2,2,1,59,79,2,2
3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,7,3,41.9,0,Groceries,Mastercard,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.44,0.13,0.7,0.67,3,2,2,2,0,3,3,0,3,3,0,2,0,3,3,3,3,5,251,338,3,3
4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,4,4,48.0,0,Electronics,Visa,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.85,0.80,0.8,0.69,4,3,2,3,3,4,4,0,4,4,3,3,0,4,4,2,4,5,42,54,4,4


In [19]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 50 columns):
 #   Column                                           Non-Null Count   Dtype   
---  ------                                           --------------   -----   
 0   amount_usd                                       699960 non-null  float64 
 1   hours_since_last_txn                             699960 non-null  float64 
 2   txn_count_last_24h                               699960 non-null  int8    
 3   distance_from_home_km                            699960 non-null  float64 
 4   card_age_months                                  699960 non-null  int8    
 5   customer_age                                     699960 non-null  int8    
 6   account_balance_usd                              699960 non-null  float64 
 7   cvv_retry_count                                  699960 non-null  int8    
 8   velocity_score                                   699960 non-null  float64 
 9   time

In [20]:
# corr_type = ['pearson', 'spearman'][0]

# plt.figure(figsize=(15, 4))
# corr_df = X.corrwith(train_X[TARGET], method=corr_type).sort_values()
# corr_df.abs().plot(kind='bar', color=[CRIM if c < 0 else BLUE for c in corr_df.values])
# for i, v in enumerate(corr_df.values):
#     plt.text(
#         i, np.abs(v) + 0.02, np.round(v, 3),
#         size=7, weight='semibold', color=CRIM if v < 0 else BLUE,
#         va='bottom', ha='center', rotation=90,
#     )

# plt.title(f"{corr_type}: Correlation ({TARGET}) rate")
# plt.show()

# ML TRAINING

In [21]:
try:
    import optuna
except ModuleNotFoundError:
    !pip install optuna -q
    import optuna

print(f"optuna version: {optuna.__version__}")

optuna version: 5.0.0


In [22]:
# try:
#     import optuna_dashboard
# except ModuleNotFoundError:
#     !pip install optuna-dashboard -q
#     import optuna_dashboard

# pip install jupyterlab-optuna

In [23]:
def objective(trial):
    FOLDS = 5
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

    # Best OOF AUC: 0.95047
    # Best Params: {'top_rate': 0.4552802308879337, 'other_rate': 0.11108320227437572,
    #               'max_depth': 2, 'num_leaves': 3, 'subsample': 0.5585528062048921,
    #               'colsample_bytree': 0.9156867345218691, 'min_child_samples': 83,
    #               'reg_alpha': 0.37709391428365335, 'reg_lambda': 0.005014722507918812,
    #               'learning_rate': 0.041778607440798296, 'smooth': 5}

    # Best OOF AUC: 0.95047
    # Best Params: {'max_depth': 3, 'num_leaves': 4, 'subsample': 0.7569021722304632,
    #               'colsample_bytree': 0.4839900563827455, 'min_child_samples': 49,
    #               'bagging_freq': 4, 'reg_alpha': 8.507284412682718e-07,
    #               'reg_lambda': 0.05946179298298863, 'learning_rate': 0.02255660573426721, 'smooth': 50}

    params = {
        # ------------------------------------------------
        'metric': 'auc',
        'n_jobs': -1,
        'verbose': -1,
        'importance_type': 'gain',
        # 'feature_pre_filter': False,
        "device": "gpu" if torch.cuda.is_available() else "cpu",
        # ------------------------------------------------
        # 'data_sample_strategy': "goss",
        # 'top_rate': trial.suggest_float('top_rate', 0.4, 0.7),
        # 'other_rate': trial.suggest_float('other_rate', 0.1, 0.25),
        # ------------------------------------------------
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
        'max_depth': trial.suggest_categorical('max_depth', [-1, 2, 4, 6]),
        'num_leaves': trial.suggest_int('num_leaves', 3, 15),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 150),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
        'max_bin': trial.suggest_int('max_bin', 255, 1024),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.05),
        # 'learning_rate': 0.05,
        'n_estimators': 20_000,
    }

    smooth = {'smooth': trial.suggest_categorical('smooth', ['auto', 5, 10, 20, 50, 100])}

    class CFG:
        TE = True if 'combo_names' in globals() else False

    fold_scores = []
    _bins = [c for c in X.columns if c.endswith('_bin_')]
    _cats = [c for c in X.columns if c.endswith('_cat_')]

    print(f"\n### Fold:")
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_trn, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_trn, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tst = X_test.copy()

        print(f"{fold}", end=". ")

        # Target encoding
        if CFG.TE:
            te_cols = combo_names + _bins + _cats
            # nums_to_cats + INTER # + floor_cols + digit_cols + bin_cols

            TE = skTE(target_type='binary', cv=5, **smooth, random_state=42)
            tr_enc = TE.fit_transform(X_trn[te_cols], y_trn)
            va_enc = TE.transform(X_val[te_cols])
            ts_enc = TE.transform(X_tst[te_cols])
    
            sm = -1 if smooth['smooth'] == 'auto' else smooth['smooth']
            te_names = [f"TE{sm}_{col}" for col in te_cols]
            X_trn[te_names] = tr_enc
            X_val[te_names] = va_enc
            X_tst[te_names] = ts_enc
    
            combos_to_drop = te_cols
            X_trn = X_trn.drop(columns=combos_to_drop)
            X_val = X_val.drop(columns=combos_to_drop)
            X_tst = X_tst.drop(columns=combos_to_drop)
        # ------------------------------------------------------------------------------
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_trn, y_trn, 
            eval_set=[(X_trn, y_trn), (X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                # lgb.log_evaluation(period=CFG.VERB),
            ]
            # init_score=trn_margin,
        )

        val_preds = model.predict_proba(X_val)[:, 1]
        fold_score = roc_auc_score(y_val, val_preds)
        fold_scores.append(fold_score)

    avg_auc = np.round(np.mean(fold_scores), 5)
    std_auc = np.round(np.std(fold_scores), 5)
    # print("-"*50)
    print(f"-AUC: {Fore.YELLOW}{Style.BRIGHT}{avg_auc} ± {std_auc}{Style.RESET_ALL}")

    return avg_auc

print("Optuna search function ready!")

Optuna search function ready!


In [24]:
# !rm -r /kaggle/working

In [25]:
# ===================
# RUN OPTUNA SEARCH
# ===================
storage = "sqlite:///db.sqlite3" # Specify the storage URL here.
st_name = "lgb_mlp_study"

opt_study = optuna.create_study(direction="maximize", storage=storage, study_name=st_name, load_if_exists=True)
opt_study.optimize(objective, n_trials=150, timeout=int(3600*11.5), show_progress_bar=True)

print("\n💥 Optimization Finished!")
print(f"Best OOF AUC: {opt_study.best_value:.5f}")
print("Best Params:", opt_study.best_params)

[I 2026-11-27 02:13:32,170] A new study created in RDB with name: lgb_mlp_study


  0%|          | 0/150 [00:00<?, ?it/s]


### Fold:
1. Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1917]	training's auc: 0.954353	valid_1's auc: 0.949489
2. Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1312]	training's auc: 0.952585	valid_1's auc: 0.951092
3. Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1437]	training's auc: 0.951939	valid_1's auc: 0.954872
4. Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1241]	training's auc: 0.952348	valid_1's auc: 0.950777
5. Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1749]	training's auc: 0.955459	valid_1's auc: 0.944691
-AUC: 0.95018 ± 0.00328
[I 2026-11-27 02:26:38,708] Trial 0 finished with value: 0.95018 and parameters: {'bagging_freq': 5, 'max_depth': -1, 'num_leaves': 4, 'subsample': 0.5125732646950188, 'colsample_bytree

In [26]:
if 'opt_study' in globals():
    print(f"Best OOF AUC: {opt_study.best_value:.5f}")
    print("Best Params:", opt_study.best_params)

Best OOF AUC: 0.95054
Best Params: {'bagging_freq': 3, 'max_depth': 2, 'num_leaves': 3, 'subsample': 0.9876780467525299, 'colsample_bytree': 0.3607420460620429, 'min_child_samples': 29, 'reg_alpha': 0.09021814296633397, 'reg_lambda': 0.0005842556056814604, 'max_bin': 429, 'learning_rate': 0.023671241977449336, 'smooth': 100}


In [27]:
## -- Visualize parameter importance
if 'opt_study' in globals():
    optuna.visualization.plot_param_importances(opt_study).show()
    optuna.visualization.plot_slice(opt_study).show()
    optuna.visualization.plot_optimization_history(opt_study).show()

In [28]:
# class CFG:
#     TE = True if 'combo_names' in globals() else False
#     FOLDS = 5
#     SEED  = 42

# FOLDS = 5
# skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

# model_name = "LGBbag_mlp"
# print(f"🚀 Training {model_name}({X.shape[1]}) with {FOLDS } Folds\n")

# oof_preds  = np.zeros(len(X))
# test_preds = np.zeros(len(X_test))

# fold_times = []
# fold_scores = []
# best_iterations = []
# feat_importances = []

# _bins = [c for c in X.columns if c.endswith('_bin_')]
# _cats = [c for c in X.columns if c.endswith('_cat_')]

# t0 = time()
# for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     X_trn, X_val = X.iloc[tr_idx], X.iloc[val_idx]
#     y_trn, y_val = y.iloc[tr_idx], y.iloc[val_idx]
#     X_tst = X_test.copy()

#     f0 = time()
#     # Target encoding
#     if CFG.TE:
#         te_cols = combo_names + _bins + _cats
#         # te_cols = nums_to_cats + INTER # + floor_cols + digit_cols + bin_cols
#         smooth = 50
#         print(f"⚡️ TargetEncoding ({len(te_cols)}) • smooth={smooth}")
#         TE = skTE(target_type='binary', cv=5, smooth=smooth, random_state=42)
#         tr_enc = TE.fit_transform(X_trn[te_cols], y_trn)
#         va_enc = TE.transform(X_val[te_cols])
#         ts_enc = TE.transform(X_tst[te_cols])
        
#         sm = -1 if smooth == 'auto' else smooth
#         te_names = [f"TE{sm}_{col}" for col in te_cols]
#         X_trn[te_names] = tr_enc
#         X_val[te_names] = va_enc
#         X_tst[te_names] = ts_enc

#         combos_to_drop = te_cols
#         X_trn = X_trn.drop(columns=combos_to_drop)
#         X_val = X_val.drop(columns=combos_to_drop)
#         X_tst = X_tst.drop(columns=combos_to_drop)
#     # ----------------------------------------------------------------
#     if fold == 1: print(f"train={X_trn.shape}, valid={X_val.shape}\n")
#     if fold == 1: display(X_trn.head())

#     print("="*15)
#     print(f"### Fold {fold}/{FOLDS}")
#     print("="*15)
#     seeder = 42 #+ (fold-1)

#     # Best OOF AUC: 0.94997
#     bagging_params = {
#         'max_depth': 3, 'num_leaves': 4, 'subsample': 0.7569021722304632,
#         'colsample_bytree': 0.4839900563827455, 'min_child_samples': 49,
#         'bagging_freq': 4, 'reg_alpha': 8.507284412682718e-07,
#         'reg_lambda': 0.05946179298298863, 'learning_rate': 0.02255660573426721,
#         # 'smooth': 50,
#     }

#     goss_params = {
#         'top_rate': 0.4552802308879337, 'other_rate': 0.11108320227437572,
#         'max_depth': 2, 'num_leaves': 3, 'subsample': 0.5585528062048921,
#         'colsample_bytree': 0.9156867345218691, 'min_child_samples': 83,
#         'reg_alpha': 0.37709391428365335, 'reg_lambda': 0.005014722507918812,
#         'learning_rate': 0.041778607440798296,
#         # 'smooth': 5,
#     }
    
#     params = {
#         **bagging_params,
#         'metric': 'auc',
#         'n_jobs': -1,
#         'verbose': -1,
#         'importance_type': 'gain',
#         # 'feature_pre_filter': False,
#         "device": "gpu" if torch.cuda.is_available() else "cpu",
#         # ------------------------------------------------
#         'n_estimators': 20_000,
#     }
#     model = lgb.LGBMClassifier(**params)
#     model.fit(
#         X_trn, y_trn, 
#         eval_set=[(X_trn, y_trn), (X_val, y_val)],
#         callbacks=[
#             lgb.early_stopping(stopping_rounds=500),
#             lgb.log_evaluation(period=500),
#         ]
#         # init_score=trn_margin,
#     )
#     feat_importances.append(np.asarray(model.feature_importances_))
    
#     val_preds = model.predict_proba(X_val)[:, 1]
#     oof_preds[val_idx] = val_preds

#     test_preds += model.predict_proba(X_tst)[:, 1]

#     fold_score = np.round(roc_auc_score(y_val, val_preds), 5)
#     fold_scores.append(fold_score)

#     best_iter = model.best_iteration_
#     best_iterations.append(best_iter)

#     f1 = time()
#     ff = np.round((f1-f0)/60, 2)
#     fold_times.append(ff)

#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

#     print(f"{Fore.YELLOW}{Style.BRIGHT}# Fold {fold} AUC: {fold_score}{Style.RESET_ALL}", end=' ')
#     print(f"[ seed= {seeder}, tree={best_iter}, time={ff} mins ]\n")

# test_preds /= FOLDS

# model_name = model_name + f"({X_trn.shape[1]})"
# feat_names = X_trn.columns

# print(f"\n{'='*50}")
# print(f"{FOLDS}-FOLD CV: {model_name} | {X_trn.shape}")
# print(f"{'='*50}")

# for i, (score, tree, dura) in enumerate(zip(fold_scores, best_iterations, fold_times), 1):
#     print(f" • Fold {i} AUC: {score} @ tree={tree} in {dura} mins")

# ## -- Final out-of-fold score --
# oof_auc = np.round(roc_auc_score(y, oof_preds), 5)
# avg_auc = np.round(np.mean(fold_scores), 5)
# std_auc = np.round(np.std(fold_scores), 5)

# ## -- CV results --
# fold_results = {}
# fold_results[f'auc'] = fold_scores
# fold_results[f'tree'] = best_iterations
# fold_results[f'time(min)'] = fold_times

# print("-"*50)
# print(f"OOF score: {Fore.YELLOW}{Style.BRIGHT}{oof_auc}{Style.RESET_ALL}")
# print(f"AVG score: {Fore.YELLOW}{Style.BRIGHT}{avg_auc} ± {std_auc}{Style.RESET_ALL}")
# print("-"*50)

# t1 = time()
# tt = (t1-t0)/60
# print(f"{tt:.2f} mins\n")

# del X_trn, y_trn

In [29]:
# ==================================================
# 5-FOLD CV: LGB_mlp(50) | (559968, 50)
# ==================================================
#  • Fold 1 AUC: 0.94627 @ tree=157 in 0.6mins
#  • Fold 2 AUC: 0.94918 @ tree=131 in 0.55mins
#  • Fold 3 AUC: 0.95304 @ tree=94 in 0.5mins
#  • Fold 4 AUC: 0.94952 @ tree=154 in 0.57mins
#  • Fold 5 AUC: 0.94171 @ tree=161 in 0.59mins
# --------------------------------------------------
# OOF score: 0.94769
# AVG score: 0.94794 ± 0.00379
# --------------------------------------------------
# 2.84 mins

In [30]:
# n = f"{model_name}_{str(avg_auc).split('.')[1]}"

# print('-'*25)
# print(f"{Fore.YELLOW}{Style.BRIGHT}{n}{Style.RESET_ALL}")
# print('-'*25)

# fold_df = pd.DataFrame(fold_results, index=[f"fold {i+1}" for i in range(FOLDS)])
# print(f"{tt:.3g} mins\n")
# fold_df #.style.background_gradient(subset=['auc'], cmap='Blues')

In [31]:
# submit[TARGET] = test_preds

# submit.to_csv(f"submit_{n}.csv", index=False)
# np.save(f"oof_{n}.npy", oof_preds)
# np.save(f"test_{n}.npy", test_preds)

# submit.head()

In [32]:
# !rm -r /kaggle/working

In [33]:
# ## -- Plot oof distributions --
# _, axs = plt.subplots(1, 3, figsize=(13, 4))

# CalibrationDisplay.from_predictions(y, oof_preds, n_bins=10, name=n, ax=axs[0])
# axs[0].set_title(f"Calibration")

# RocCurveDisplay.from_predictions(y, oof_preds, name=n, ax=axs[1])
# axs[1].set_title(f"ROC-Curve")

# sns.histplot(oof_preds, ax=axs[2], bins=50)
# axs[2].set_title(f"Distribution")

# plt.tight_layout(pad=0.2)
# plt.show()

In [34]:
# ## -- Get model feature importances --
# top_n = 30

# fi_df = pd.DataFrame({"feature": feat_names, "score": np.mean(feat_importances, axis=0),})\
#           .sort_values(by="score", ascending=False).reset_index(drop=True)

# plt.figure(figsize=(13, top_n*0.25))
# sns.barplot(data=fi_df.head(top_n), x="score", y="feature", palette='Blues_r')

# plt.title(f"{n}: Avg. {FOLDS}-Fold feature importances", loc='left')
# plt.tight_layout()
# plt.show()

# print()

In [35]:
# try:
#     import shap
# except:
#     !uv pip install -q shap
#     import shap

# print(f"Shap version: {shap.__version__}")

In [36]:
# print(f"Model pipeline features: {len(feat_names)}")

# # Randomly select 100 rows without replacement
# rng = np.random.default_rng(seed=139)
# indices = rng.choice(X_val.shape[0], size=100, replace=False)
# val_x = X_val.to_numpy()[indices]
# val_y = y_val.to_numpy()[indices]

# ## -- Shap Explainer --
# def f(x):
#     return model.predict_proba(x)[:, 1]

# val_y

In [37]:
# _id = 0
# MAX = 20

# explainer   = shap.Explainer(f, val_x, feature_names=feat_names, seed=42)
# explanation = explainer(val_x, max_evals=max(500, 2*val_x.shape[1]+1))

# ## -- Bar plot --
# plt.subplot(121)
# shap.plots.bar(explanation[_id], max_display=MAX, show=False)
# plt.title(f"Impact on model predictions for label {val_y[_id]}")

# ## -- Beeswarm plot --
# plt.subplot(122)
# shap.plots.beeswarm(
#     explanation,
#     order=explanation.abs.max(0),
#     plot_size=(13, 8),
#     max_display=MAX,
#     color_bar=False,
#     show=False,
# )

# plt.title("Beeswarm - Most impactful features")
# plt.suptitle(f"{n}: SHAP Explainer", size=12, weight="semibold", color=CRIM)
# plt.tight_layout(pad=1.5)
# plt.show()

# print()